# Gold Hospitality Hiring Mart

**Purpose**: Hospitality sector-specific hiring trends for specialized dashboards.

**Target Table**: `workspace.gold.gold_hospitality_hiring`

**Grain**: One row per date with hospitality sector aggregations by role and location

**Sector Filter**: Filters to hospitality sector_sk values only

**Key Metrics**:
- Active hospitality jobs
- New hospitality postings
- Job closures
- Net hiring change
- Role and location distribution

**Data Sources**:
- `workspace.warehouse.fact_job_postings` (filtered to hospitality sector)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_hospitality_hiring
USING DELTA
COMMENT 'Hospitality sector-specific hiring trends'
AS

WITH hospitality_sector AS (
  SELECT sector_sk
  FROM workspace.warehouse.dim_sector
  WHERE sector_name IN ('Hospitality', 'Hotels & Resorts', 'Restaurants', 'Food & Beverage')
     OR sector_family = 'Hospitality'
),

base_metrics AS (
  SELECT
    f.posting_date_sk AS hospitality_date_sk,
    f.sector_sk,
    f.role_sk,
    f.location_sk,
    
    -- MEASURES
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) - 
      COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS net_change,
    
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS unique_companies,
    
    -- Remote work metrics
    COUNT(CASE WHEN f.active_flag = TRUE AND j.remote_type = 'REMOTE' THEN 1 END) AS remote_jobs,
    COUNT(CASE WHEN f.active_flag = TRUE AND j.remote_type = 'ONSITE' THEN 1 END) AS onsite_jobs
    
  FROM workspace.warehouse.fact_job_postings f
  INNER JOIN hospitality_sector hs ON f.sector_sk = hs.sector_sk
  LEFT JOIN workspace.warehouse.dim_job j ON f.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY f.posting_date_sk, f.sector_sk, f.role_sk, f.location_sk
),

-- Aggregation: Daily hospitality overview (all roles, all locations)
hospitality_daily AS (
  SELECT
    hospitality_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    'daily_total' AS aggregation_level,
    SUM(active_jobs) AS active_jobs,
    SUM(new_jobs) AS new_jobs,
    SUM(closed_jobs) AS closed_jobs,
    SUM(net_change) AS net_change,
    MAX(unique_companies) AS unique_companies,
    SUM(remote_jobs) AS remote_jobs,
    SUM(onsite_jobs) AS onsite_jobs
  FROM base_metrics
  GROUP BY hospitality_date_sk, sector_sk
),

-- Aggregation: By role within hospitality
hospitality_by_role AS (
  SELECT
    hospitality_date_sk,
    sector_sk,
    role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    'by_role' AS aggregation_level,
    SUM(active_jobs) AS active_jobs,
    SUM(new_jobs) AS new_jobs,
    SUM(closed_jobs) AS closed_jobs,
    SUM(net_change) AS net_change,
    MAX(unique_companies) AS unique_companies,
    SUM(remote_jobs) AS remote_jobs,
    SUM(onsite_jobs) AS onsite_jobs
  FROM base_metrics
  GROUP BY hospitality_date_sk, sector_sk, role_sk
),

-- Aggregation: By location within hospitality
hospitality_by_location AS (
  SELECT
    hospitality_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    location_sk,
    'by_location' AS aggregation_level,
    SUM(active_jobs) AS active_jobs,
    SUM(new_jobs) AS new_jobs,
    SUM(closed_jobs) AS closed_jobs,
    SUM(net_change) AS net_change,
    MAX(unique_companies) AS unique_companies,
    SUM(remote_jobs) AS remote_jobs,
    SUM(onsite_jobs) AS onsite_jobs
  FROM base_metrics
  GROUP BY hospitality_date_sk, sector_sk, location_sk
),

combined_agg AS (
  SELECT * FROM hospitality_daily
  UNION ALL
  SELECT * FROM hospitality_by_role
  UNION ALL
  SELECT * FROM hospitality_by_location
),

temporal_enriched AS (
  SELECT
    ca.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(ca.active_jobs, 7) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.location_sk
      ORDER BY ca.hospitality_date_sk
    ) AS prev_week_active_jobs,
    
    LAG(ca.active_jobs, 30) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.location_sk
      ORDER BY ca.hospitality_date_sk
    ) AS prev_month_active_jobs,
    
    -- Week-to-date cumulative
    SUM(ca.new_jobs) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.location_sk,
        YEAR(TO_DATE(CAST(ca.hospitality_date_sk AS STRING), 'yyyyMMdd')),
        WEEKOFYEAR(TO_DATE(CAST(ca.hospitality_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY ca.hospitality_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wtd_new_jobs,
    
    -- Month-to-date cumulative
    SUM(ca.new_jobs) OVER (
      PARTITION BY ca.aggregation_level, ca.sector_sk, ca.role_sk, ca.location_sk,
        YEAR(TO_DATE(CAST(ca.hospitality_date_sk AS STRING), 'yyyyMMdd')),
        MONTH(TO_DATE(CAST(ca.hospitality_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY ca.hospitality_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs
    
  FROM combined_agg ca
)

SELECT
  -- DIMENSIONS
  te.hospitality_date_sk,
  te.sector_sk,
  te.role_sk,
  te.location_sk,
  te.aggregation_level,
  
  -- MEASURES
  te.active_jobs,
  te.new_jobs,
  te.closed_jobs,
  te.net_change,
  te.unique_companies,
  te.remote_jobs,
  te.onsite_jobs,
  
  -- TEMPORAL METRICS
  te.wtd_new_jobs,
  te.mtd_new_jobs,
  
  -- DERIVED: Percent changes
  CASE 
    WHEN te.prev_week_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_week_active_jobs) AS DECIMAL(10,4)) / te.prev_week_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_week,
  
  CASE 
    WHEN te.prev_month_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_month_active_jobs) AS DECIMAL(10,4)) / te.prev_month_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_month,
  
  -- DERIVED: Remote work ratio
  CASE 
    WHEN te.active_jobs > 0 
    THEN CAST(te.remote_jobs AS DECIMAL(10,4)) / te.active_jobs
    ELSE NULL 
  END AS remote_ratio,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
ORDER BY te.aggregation_level, te.hospitality_date_sk DESC;

In [0]:
%sql
-- Validation: Hospitality hiring summary
SELECT
  aggregation_level,
  COUNT(*) AS row_count,
  MIN(hospitality_date_sk) AS min_date,
  MAX(hospitality_date_sk) AS max_date,
  ROUND(AVG(active_jobs), 2) AS avg_active_jobs,
  ROUND(AVG(new_jobs), 2) AS avg_new_jobs,
  ROUND(AVG(remote_ratio), 4) AS avg_remote_ratio
FROM workspace.gold.gold_hospitality_hiring
GROUP BY aggregation_level
ORDER BY aggregation_level;